# Models for multimodal data integration

**Authors: Jana Braunger & Britta Velten**

In this notebook, you will learn how to use some of the multi model data integration methods discussed in the lecture. It consists of three main parts:
1. Application of MOFA to a multi-omics data set (gene expression, methylation, mutations and drug responses)
2. Application of MEFISTO to a longitudinal data set (gene expression measurement samples over developmental time in multiple species and organs)
3. Using pyro to implement your own factor model

First, we need to install some additional packages:

In [ ]:
!ls


In [ ]:
!pip install dtw-python
!pip install scanpy==1.9.3
!pip install muon
!pip install mafax
!pip install mofapy2
!pip install pyro-ppl

# Application of MOFA to a multi-omics data set

The following section shows how to use the `muon` package to apply the factor model MOFA to integrate multi-omics data. The code is based on [this](https://muon-tutorials.readthedocs.io/en/latest/CLL.html) tutorial. We apply MOFA to a multi-omics data set, in which various data modalities have been measured in primary cancer cells of 200 individual patients with chronic lymphocytic leukemia (CLL). For more details on the data please see a detailed analysis in the [original publication](https://www.huber.embl.de/pub/pdf/Dietrich2018.pdf).

In [ ]:
# Required packages
import scanpy as sc
import muon as mu
import numpy as np
import pandas as pd
import mofax as mofa
import seaborn as sns

## Load data

Count matrices and metadata of the multi-omics CLL study are publicly available as part of the [MOFAdata R package](http://bioconductor.org/packages/release/data/experiment/html/MOFAdata.html). For the purpose of this notebook, those matrices were saved in individual `.csv` files in the `data/` folder. Please make sure to add this folder to your local directory. The data set consists of the following four modalities:  
1. mRNA: normalized expression of 5000 variable genes
2. methylation: methylation state of 4248 CpG locations
3. mutations: occurrence of insertions or deletions at 69 DNA regions
4. drugs: ex vivo drug sensitivity measurements for 310 drugs

In [ ]:
# Create dictionary of AnnData objects (one per modality)
data_dir = "data/"
mods = {i:sc.AnnData(pd.read_csv(f"{data_dir}{i}.csv", index_col=0).T)
        for i in ("mRNA", "methylation", "mutations", "drugs")}
mods

The data per modality is stored in the `AnnData` format (more infos about the Anndata structure [here](https://anndata.readthedocs.io/en/latest/)). This format is typically also used in single cell analysis, e.g. by the package `scanpy`. The value matrices can be accessed using `.X`, the row names are stored under `.obs_names` and the column names under `.var_names`. Let's explore how the input data of the mRNA modality looks like:

In [ ]:
print("First sample names: ", mods['mRNA'].obs_names[0:5].tolist())
print("First gene names: ", mods['mRNA'].var_names[0:5].tolist())
print("Values: ", mods['mRNA'].X[0:5, 0:5])
print("Mean: ", np.mean(mods['mRNA'].X[[0]]), ", Variance: ", np.var(mods['mRNA'].X[[0]]))

**Excercise:** Explore how the data matrices of the other modalities look like. Based on this, which likelihood distributions would you choose for each modality for training a MOFA model?

Next, we are creating a data frame with some additional information about each patient which includes:
- Gender: m (male), f (female)
- Age: age in years
- TTT: time (in years) between taking the sample to the next treatment
- TTD: time (in years) between taking the sample to patients' death
- treatedAfter: True/False indicating whether patient has been treated
- Died: True/False indicating whether the patient died

In [ ]:
# load meta data information
CLL_metadata = pd.read_csv("ftp://ftp.ebi.ac.uk/pub/databases/mofa/cll_vignette/sample_metadata.txt",
                           sep="\t", index_col="sample")
CLL_metadata.head()

To create a multi-modal data object from the dictionary containing the four single modalities, we use the package `muon`. `muon` allows to directly run MOFA on this `MuData` object. If you want to learn more about `muon`, check out its [documentation](https://muon.readthedocs.io/en/latest/notebooks/quickstart_mudata.html) or [paper](https://link.springer.com/article/10.1186/s13059-021-02577-8).

In [ ]:
mdata = mu.MuData(mods)
# add metadata
mdata.obs = mdata.obs.join(CLL_metadata)
mdata

## Integrate data

**Multi-omics factor analysis** (MOFA) integration can be run on a MuData object with a single command:

```
mu.tl.mofa()
```

In case some samples are missing from some of the modalities, as they are in this CLL dataset, `muon` can be instructed to handle it. One of two modes has to be chosen for that:

1. `use_obs='union'`, which will populate values for missing samples in certain modalities with NAs, or

1. `use_obs='intersection'`, which will use a subset of samples that are present in all modalities.

We will use the first option here. Additionally, we have to specify how many latent factors should be inferred which is set to 15 in this example. With these settings we can now train a MOFA model which takes a few minutes to run.

In [ ]:
mu.tl.mofa(mdata, use_obs='union',
           n_factors=15, convergence_mode='medium',
           outfile="models/CLL.hdf5")

## Interpret intergrated data

Let's have a look at the model output and analyse what is captured in the latent factors. For this, we are using the `mofax` package to load our trained model and then plot the amount of variance that each factor explains across the data modalities.

In [ ]:
# load trained MOFA model
model = mofa.mofa_model("models/CLL.hdf5")
model

In [ ]:
# variance decomposition
mofa.plot_r2(model, x='View', vmax=15)

We notice e.g. that `Factor 1` captures variability across all modalities, whereas `Factor 2` and `Factor 4` are quite specific to one modality, drugs and mRNA respectively.

**Excercise**: Train another MOFA model using only 5 factors. How much of the total variance per view can be explained with the new model compared to the existing one? Hint: use the function model.get_r2()

Next, we can plot our samples in the learned latent space. For this, we can use the `plot_factors` function which allows to choose the latent factors to plot, as well as a covariate from the metadata for coloring.   

In [ ]:
# Plotting the samples in the learned factor space
mofa.plot_factors(model, x = "Factor1", y="Factor2",  size=20, color = "Gender")

Factor 1 seems to separate the patients into 2 groups. Since males and females are well mixed in this plot, variability between gender doesn't seem to be captured in Factor 1 nor Factor 2.

**Excercise:** How does the plot look like for other factors? Try out some combinations of factors and other coloring choices to check whether some of the covariates are linked to the values of a factor. (`model.metadata` can be used to look at the available metadata information for coloring)

### Factor 1

Let's explore which features are driving the differences across patients in factor 1. For this, we look at the feature weights in the MOFA model and label the ones with highest absolute weights.

In [ ]:
mofa.plot_weights(model, views=['mRNA', 'mutations', 'drugs'], factors=0, zero_line=True, ncols=3, label_size=10);

We can see that in the mutations modality IGHV has a much higher absolute weight than all other features. Additionally, the top weights from the RNA modality are likely to represent the same latent process as IGHV status:

- ENSG00000198046 &rarr; ZNF667 — lnc‐ZNF667‐AS1 [is associated with poor survival and is a promising prognostic biomarker](https://onlinelibrary.wiley.com/doi/abs/10.1111/ijlh.13167)

- ENSG00000186522 &rarr; SEPT10 – its expression [correlates with IGHV status](https://www.nature.com/articles/2404220) (however [there are some caveats](https://www.nature.com/articles/2404867))

- ENSG00000198142 &rarr; SOWAHC – it [has been identified as a prognostic marker for other cancers](https://www.spandidos-publications.com/mmr/21/3/1285)

Focusing on the drug response, we notice PF477736 (D_078), AZD7762 (D_020), AT13387 (D_017), and dasatinib (D_050) have the major association with this factor. Those are the exact top associations with the IGHV status [described in the original paper](https://www.huber.embl.de/pub/pdf/Dietrich2018.pdf).

Let's check how the factor plot we created before looks like when we color it according to IGHV status.

In [ ]:
# Coloring the samples according to IGHV status
model.metadata.IGHV = (
    model.metadata.IGHV.astype(str).
        replace({'1.0': 'mutated', '0.0': 'unmutated'}).
        astype('category').cat.reorder_categories(["mutated", "unmutated", "nan"])
)

mofa.plot_factors(model, x = "Factor1", y="Factor2",  size=20, color = "IGHV")

So, the two clusters along the Factor1 axis nicely separate the patients based on whether they have the IGHV mutation and this seems the major factor of variation across patients in this CLL dataset.

### Factor 3

Next, we like to perform a similar analysis for factor 3 since this factor also captures variability which is shared across multiple modalities.   
**Excercise:** Explore the weights in the different modalities of factor 3 to figure out its biological interpretation (remember to use `factors = 2` since the counting starts at 0).

Short side note: in addition to the `get_factors` function, the mofax package contains the `get_weights` function to obtain the weight matrix for specified latent factors. These function allow to extract the inferred matrices of the model in case one likes to perform any additional user-specific analyses.

### Combinations of factors

Summarizing our results of what we have learned from interpreting factor 1 and 3 values, it seems like that a couple of major factors associated with differences in the drug response assay seem to capture molecular profiles of patients. Therefore, it might be useful for better understanding the drug response in CLL patients, to group patients according to their somatic mutations status (IGHV and trisomy12).

In [ ]:
mofa.plot_factors(model, x=0, y=2, color=["IGHV", "trisomy12"], size=20,  palette="Set2");

# Application of MEFISTO to a longitudinal data set

Currently, an increasing number of multi omics data sets is created in which the samples are not indepently from each other but contain some known relationships between them. Examples for this would be collecting samples over multiple time points or in a spatial manner. For this purpose, the MOFA extension MEFISTO which is based on Gaussian Processes (for details see the lecture material) has been developed. In the following section, we are going to apply MEFISTO to an evodevo dataset in which gene expression has been measured in samples from 5 different species in 5 organs across developmental time (from embryonic development to adulthood). The code of this section is based on [this](https://muon-tutorials.readthedocs.io/en/latest/mefisto/1-MEFISTO-evodevo.html) tutorial.

## Load data

First we will load the evodevo data containing normalized gene expression data for the 5 species (groups of the model: Human, Mouse, Rat, Rabbit and Opossum) and 5 organs (views of the model: Brain, Cerebellum, Heart, Liver and Testis) as well as the developmental time information for each sample. Note that in this data set only one modality (gene expression) has been measured, but we are instead treating the organs as different views of the model in this scenario. Please download the `evodevo.csv` data [from here](https://figshare.com/s/242916198fde3353f3e6) and save it in your data folder or use wget.


In [ ]:
data = pd.read_csv("data/evodevo.csv", sep=",", index_col=0)
data

As you can see in the above data frame, the samples are here time points in various species at which a measurement has been taking during their development (wpc = weeks post conception). Since multiple species are used, we use the multi-group framework of MOFA, in which a group-specific factor matrix is learned.

To read the data into muon, we again first create a dictionary containing an AnnData object for each organ. Since the data is in long format, we first convert it to a wide data frame (samples x features). For instance, here's the data for the `'Brain'` view:

In [ ]:
brain = data[data.view == 'Brain'].pivot(index='sample', columns='feature', values='value')
brain

In [ ]:
# Create dictionary of AnnData objects per view
views = data.view.unique()
data_list = [data[data.view == m].pivot(index='sample', columns='feature', values='value') for m in views]
mods = {views[m]:sc.AnnData(data_list[m]) for m in range(len(views))}
mods

In [ ]:
# Metadata: stores time and group (= species) information for each sample
obs = (
    data[['sample', 'time', 'group']]
        .drop_duplicates()
        .rename(columns = {'group' : 'species'})
        .set_index('sample')
)
obs

In [ ]:
# Create mudata object
mdata = mu.MuData(mods)
mdata.obs = mdata.obs.join(obs)

## Integrate data

**MEFISTO** can be run on a `MuData` object with `mu.tl.mofa` by specifying which variable (*covariate*) should be treated as *time*.

- To incorporate the time information, we specify which metadata column to use as a covariate for MEFISTO — `'time'`.

- We also specify `'species'` to be used as groups.

- In addition, we tell the model that we want to learn an alignment of the time points from different species by setting `smooth_warping=True` and using `'Mouse'` as reference.

- Using the underlying Gaussian process for each factor we can interpolate to unseen time points. This is enabled by providing `new_values`, which correspond to the covariate, i.e. time.

For illustration, we only use a small number of training iterations.

In [ ]:
mu.tl.mofa(mdata, n_factors=5,
           groups_label="species",
           smooth_covariate='time', smooth_warping=True,
           smooth_kwargs={"warping_ref": "Mouse", "new_values": list(range(1, 15))},
           outfile="models/mefisto.hdf5",
           n_iterations=25)

## Visualization in the factor space

Let's first plot the samples in the learned factor space and color them according to species and time.

In [ ]:
mu.pl.mofa(mdata, color="species", size = 250)

In [ ]:
mu.pl.mofa(mdata, color="time", size = 250)

In the previous plot, the samples have been colored according to the time point numbered according to the ordering of the measured samples. However, there have been different numbers of samples collected for each species and also at different time points (since embryonic development e.g. takes much longer in humans than mice). Thus, it is unclear ahead of time which human time point corresponds to what mouse time point. Therefore, MEFISTO also infers the best alignment of time points across species. We can color the above plot with the learned time of each sample in the following and see that the trajectory across the two factors matches to the time information.

In [ ]:
mu.pl.mofa(mdata, color="time_warped", size = 250)

## Alignment

We can also visualize the learned matched time points (`time_warped`) over the original time points. As you can see, for `Human` the highest number of time points (>20) has been measured, but they were then mapped to the corresponding `Mouse` time points (1-14) such that multiple human time points were mapped to the same mouse stage:

In [ ]:
sc.pl.scatter(mdata, x="time", y="time_warped", color="species", size=200)

## Latent factors versus common developmental time
We can also plot the latent processes along the inferred common developmental time.
Before alignment:

In [ ]:
mdata.obs["Factor1"] = mdata.obsm["X_mofa"][:,0]
sc.pl.scatter(mdata, x="time", y="Factor1", color="species", size=200)

In [ ]:
# after alignment
sc.pl.scatter(mdata, x="time_warped", y="Factor1", color="species", size=200)

This plot shows that `Factor1` mostly captures the variability during development which is shared across all species.

## Further analyses
Additionally we can take a look at the smoothness and sharedness of the factors, interpolate the factors or cluster the species based on the learnt group kernel of each latent factor.

For that, we will use the [mofax](https://github.com/bioFAM/mofax) library that provides convenient loaders and plotting functions.

In [ ]:
model = mofa.mofa_model("models/mefisto.hdf5")
model

### Interpolation

Using the underlying Gaussian process for each factor we can interpolate to unseen time points for species that are missing data in these time points or intermediate time points.

In [ ]:
mofa.plot_interpolated_factors(model, factors=range(model.nfactors),
                                ncols=5, size=70);

### Smoothness and sharedness of factors

In addition to the factor values and the alignment the model also inferred an underlying Gaussian process that generated these values. By looking into it we can extract information on the smoothness of each factor, i.e. how smoothly it varies along developmental time, as well as the sharedness of each factor, i.e. how much the species (groups) show the same underlying developmental pattern and how the shape of their developmental trajectory related to a given developmental module (Factor) clusters between species.

The scale parameters of the Gaussian process capture the smoothness of the model. We will visualize them with bar plots where more colour means more smoothness:

In [ ]:
mofa.plot_smoothness(model);

In [ ]:
mofa.plot_sharedness(model);

`Factor1` and `Factor2` have the highest amount of sharedness. This also fits to the observations made in the interpolated factors plot, where these factors are very similar across species, whereas `Factor3` e.g. seems to capture a opposom-specific signature over time which is different from the other species.

# Advanced exercise: using pyro to built your own factor model

In the last part of this notebook, we are using `pyro` to built our own Bayesian matrix factorization model and to get a better understanding of how a factor model can be trained using stochastic variational inference. We are implementing a simple factor model that can derive weight matrix $W$ and factor matrix $Z$ from $Y$ ($Y = Z*W^T$). We have $Y^{NxD} = Z^{NxK} * W^{(DxK)T}$ with $N$ samples, $D$ features and $K$ latent factors. We make this matrix factorization Bayesian by adding priors to the matrices $Z$ and $W$.

In [ ]:
# Required packages
import pyro
from pyro.nn import PyroSample, PyroModule
from pyro.infer import SVI, Trace_ELBO, autoguide
import torch
from torch.nn.functional import softplus
from sklearn.metrics import mean_squared_error
import random
import matplotlib.pyplot as plt
import seaborn as sns

First, we generate a random factor and weight matrix and use its product for training our model to analyse to what extent the original input can be reconstructed.

In [ ]:
# simulate data
n_obs = 100
n_features = 20
n_factors = 5

torch.manual_seed(2024)
Z_in = torch.randn(n_obs, n_factors)
W_in = torch.randn(n_features, n_factors)

# create observated values from the simulated factor and weight matrix with some random noise
Y = torch.matmul(Z_in, W_in.t()) + 0.2 * torch.randn(n_obs, n_features)
print(Y.shape)
print(Y)

**Excercises:**
1. Training a factor model with 5 latent factors:
    - Use the FA class (in the chunk below) to train a factor model with K = 5 latent factors on the simulated data Y.
    - Plot the loss over the number of steps.
    - Calculate the mean squared error between the input observations Y and an estimate Y_hat based on your factor model (by multiplying the matrix estimates of the trained model).  
2. Identifying the number of latent factors in the data:
    - Train a factor model on Y with K = 2,3,4,5,6,7 latent factors and calculate for each model the mean squared error.
    - Plot the mean squared error over the number of latent factors. Does the curve fit to your expectations given that we simulated Y with K = 5 factors?  
3. Missing values in the observations:
    - Create a modified version $Y_{missing}$ of Y in which random entries of the values in Y are missing (set to NA). Set 1, 2.5, 5, 7.5, 10, 15, 20, 25 or 30% of all entries to NA.
    - Train a factor model with K = 5 on each of these modified matrices and calculate its mean squared error.
    - Plot the mean squared error over the percentage of missing values.

In [ ]:
class FA(PyroModule):
    def __init__(self, Y, K):
        """
        Args:
            Y: Tensor (Samples x Features)
            K: Number of Latent Factors
        """
        super().__init__()
        pyro.clear_param_store()

        # data
        self.Y = Y
        self.K = K

        self.num_samples = self.Y.shape[0]
        self.num_features = self.Y.shape[1]

        self.sample_plate = pyro.plate("sample", self.num_samples)
        self.feature_plate = pyro.plate("feature", self.num_features)
        self.latent_factor_plate = pyro.plate("latent factors", self.K)


    def model(self):
        """
        how to generate a matrix
        """
        with self.latent_factor_plate:
            with self.feature_plate:
                # sample weight matrix with Normal prior distribution
                W = pyro.sample("W", pyro.distributions.Normal(0., 1.))

            with self.sample_plate:
                # sample factor matrix with Normal prior distribution
                Z = pyro.sample("Z", pyro.distributions.Normal(0., 1.))

        # estimate for Y
        Y_hat = torch.matmul(Z, W.t())

        with pyro.plate("feature_", self.Y.shape[1]), pyro.plate("sample_", self.Y.shape[0]):
            # masking the NA values such that they are not considered in the distributions
            obs_mask = torch.ones_like(self.Y, dtype=torch.bool)
            if data is not None:
                obs_mask = torch.logical_not(torch.isnan(self.Y))
            with pyro.poutine.mask(mask=obs_mask):
                if data is not None:
                    # a valid value for the NAs has to be defined even though these samples will be ignored later
                    self.Y = torch.nan_to_num(self.Y, nan=0)

                    # sample scale parameter for each feature-sample pair with LogNormal prior (has to be positive)
                    scale = pyro.sample("scale", pyro.distributions.LogNormal(0., 1.))
                    # compare sampled estimation to the true observation Y
                    pyro.sample("obs", pyro.distributions.Normal(Y_hat, scale), obs=self.Y)


    def train(self):
        # set training parameters
        optimizer = pyro.optim.Adam({"lr": 0.02})
        elbo = Trace_ELBO()
        guide = autoguide.AutoDelta(self.model)

        # initialize stochastic variational inference
        svi = SVI(
            model = self.model,
            guide = guide,
            optim = optimizer,
            loss = elbo
        )

        num_iterations = 2000
        train_loss = []
        for j in range(num_iterations):
            # calculate the loss and take a gradient step
            loss = svi.step()

            train_loss.append(loss/self.Y.shape[0])
            if j % 200 == 0:
                print("[iteration %04d] loss: %.4f" % (j + 1, loss / self.Y.shape[0]))

        # Obtain maximum a posteriori estimates for W and Z
        map_estimates = guide(Y)

        return train_loss, map_estimates

In [ ]:
# params
k_values = {f'{k}': k for k in [2, 3, 4, 5, 6, 7]}

results = {}
for key, val in k_values.items():
    model = FA(Y, val)
    loss, result = model.train()
    results[key] = {
        'loss': loss,
        'results': result}

In [ ]:
# plot results 
for k_val, result in results.items():
    plt.plot(result['loss'], label=f'k = {k_val}')
    plt.title('Lost function results')
    plt.legend()

plt.show()

In [ ]:
# plot results 
for k_val, result in results.items():
    # calculations 
    
    # plots
    plt.plot(result['results'], label=f'k = {k_val}')
    plt.title('Lost function results')
    plt.legend()

plt.show()